In [0]:
df = spark.read.csv(
    "/Volumes/mlops_course/ingesting_data/files/DailyDelhiClimateTrain.csv",
    header=True,
    inferSchema=True
)

df_sorted = df.orderBy("date")



In [0]:
# Get unique sorted dates
unique_dates = [row.date for row in df_sorted.select("date").distinct().orderBy("date").collect()]

total_dates = len(unique_dates)
batches = 5
batch_size = total_dates // batches

for i in range(batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if i < batches - 1 else total_dates
    batch_dates = unique_dates[start_idx:end_idx]
    batch_df = df_sorted.filter(df_sorted.date.isin(batch_dates))
    batch_df = batch_df.select("date", "meantemp", "humidity", "wind_speed", "meanpressure")
    batch_df.write.mode("overwrite").option("header", True).csv(
        f"/Volumes/mlops_course/ingesting_data/stage_batch_files/batch_{i+1}"
    )
